# Level 2, Notebook 02: Compare and Match Records

**Problems in this notebook**

- Problem 13: Reconcile CRM and Data Warehouse Records
- Problem 18: Data Connector Sync Conflict Detection
- Problem 15: Duplicate Support Ticket Detection

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill in the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference solution only after you have tried.

The ideas behind these problems are in `02_compare_and_match_records_THEORY.md`, next to this notebook.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 13: Reconcile CRM and Data Warehouse Records

**The situation**

A sales assistant answers questions like "Is Acme still a customer?". The CRM says Acme is active, but the data warehouse says churned. If the AI picks one at random, the sales rep gets a confident wrong answer. You want a list of every customer where the two systems disagree.

**Your task**

Write `reconcile_records(crm, warehouse, field)`. It returns the sorted customer IDs where the chosen field differs between the two systems, including customers that exist in only one of them.

**Example**

`crm={"c1": {"status": "active"}}`, `warehouse={"c1": {"status": "churned"}}`, field `"status"` -> `["c1"]`

**Ask before coding**

- Which system is the source of truth when they disagree?
- Should a customer missing from one system count as a mismatch?
- Should `"Active "` and `"active"` count as the same value?

**Plan to say aloud**

1. Take every customer ID from both systems (the union).
2. For each ID, read the field from each side with `.get()`, using None when it is missing.
3. Clean string values: trim spaces and lowercase.
4. If the two values differ, record the ID.
5. Return the IDs sorted, so the report is the same every run.

### Prompt for your AI assistant

```text
Write a Python function `reconcile_records(crm: dict[str, dict], warehouse: dict[str, dict], field: str) -> list[str]`. Both inputs map customer_id to a record dict. Look at every customer_id that appears in either input. Read `field` from each side; a missing customer or missing field counts as None. Before comparing, strip and lowercase string values only. Return the customer IDs whose values differ, sorted ascending. Do not modify the inputs. Show only the function.
```

**Check the AI's code for**

- Loops over `crm` only, so customers that exist only in the warehouse are never reported.
- Uses `warehouse[customer_id]` and crashes when a customer is missing.
- Builds the result from a set without sorting, so the order changes between runs.
- Calls `.lower()` on every value, which crashes on numbers and None.

In [ ]:
def reconcile_records(crm: dict[str, dict], warehouse: dict[str, dict], field: str) -> list[str]:
    """Return sorted customer IDs where `field` differs between the two systems.

    Rules:
    - Check every customer_id found in either system.
    - A missing customer or missing field counts as None.
    - Strip and lowercase string values before comparing.
    - Return the IDs sorted ascending.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"c1": {"status": "active"}}, {"c1": {"status": "churned"}}, "status"), ["c1"]),
    (({"c1": {"status": "Active "}}, {"c1": {"status": "active"}}, "status"), []),
    (({"c2": {"status": "active"}}, {}, "status"), ["c2"]),
    (({"c3": {"status": "active"}, "c1": {"status": "active"}, "c2": {"status": "active"}},
      {"c3": {"status": "churned"}, "c1": {"status": "paused"}, "c2": {"status": "active"}},
      "status"), ["c1", "c3"]),
    (({"c1": {"plan": "pro"}}, {"c1": {"plan": "pro"}}, "status"), []),
    (({"c1": {"seats": 10}}, {"c1": {"seats": 12}}, "seats"), ["c1"]),
]
run_checks(reconcile_records, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def _clean(value):
    return value.strip().lower() if isinstance(value, str) else value


def reconcile_records_reference(crm: dict[str, dict], warehouse: dict[str, dict], field: str) -> list[str]:
    mismatches = []
    for customer_id in set(crm) | set(warehouse):
        crm_value = _clean(crm.get(customer_id, {}).get(field))
        wh_value = _clean(warehouse.get(customer_id, {}).get(field))
        if crm_value != wh_value:
            mismatches.append(customer_id)
    return sorted(mismatches)


assert run_checks(reconcile_records_reference, cases)

**Explain it in 30 seconds**

> "I take every customer ID from both systems, pull the field from each side, and compare after a light clean-up of spaces and case. A customer missing from one side shows up as a mismatch, because that gap is worth knowing about. I return sorted IDs so the report is stable. I only detect differences; deciding which system wins is a separate step."

**Edge cases to mention**

- A customer in only one system is reported, because None differs from a real value.
- If both sides lack the field, they match (None equals None). Worth confirming that is wanted.
- Case and spacing are ignored for strings only, so numbers are compared exactly.
- A stale value can match by luck. Spotting staleness needs timestamps.

**If they ask "how would this work in production?"**

- Agree on a source of truth per field, for example billing status always comes from the warehouse.
- Write each mismatch to a report with both values and when each was last updated, so fixes can be audited.
- Keep detection and repair separate. Never auto-overwrite a system without an approved rule.

## Problem 18: Data Connector Sync Conflict Detection

**The situation**

A connector copies documents from a file store into your RAG index. Two sync runs, or two source systems, report different version numbers for the same document. If the old one gets indexed, the assistant quotes an outdated policy.

**Your task**

Write `detect_sync_conflicts(source_a, source_b)`. It returns the sorted document IDs that appear in both maps but with different version numbers.

**Example**

`source_a={"d1": 2}`, `source_b={"d1": 3}` -> `["d1"]`

**Ask before coding**

- Is the version a number, a timestamp, or a content hash?
- Is a document found in only one source a conflict, or a separate "missing" report?
- How are deleted documents shown?

**Plan to say aloud**

1. Find the IDs that both sources have (set intersection).
2. For each shared ID, compare the two versions.
3. Collect the IDs where they differ.
4. Return them sorted.

### Prompt for your AI assistant

```text
Write a Python function `detect_sync_conflicts(source_a: dict[str, int], source_b: dict[str, int]) -> list[str]`. Both map document_id to a version number. Only compare IDs present in both maps. Return the IDs whose versions differ, sorted ascending. IDs found in only one map are not conflicts. Show only the function.
```

**Check the AI's code for**

- Uses the union of keys, then crashes with KeyError or reports one-sided documents as conflicts.
- Only flags IDs where A is older than B, missing the case where B is older.
- Forgets to sort, so list comparisons fail at random.

In [ ]:
def detect_sync_conflicts(source_a: dict[str, int], source_b: dict[str, int]) -> list[str]:
    """Return sorted document IDs present in both sources with different versions.

    Rules:
    - Only compare IDs found in both maps.
    - IDs in just one map are not conflicts.
    - Return the IDs sorted ascending.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (({"d1": 2}, {"d1": 3}), ["d1"]),
    (({"d1": 5}, {"d1": 4}), ["d1"]),
    (({"d1": 2, "d2": 5}, {"d1": 2, "d3": 1}), []),
    (({"b": 1, "a": 1, "c": 4}, {"b": 2, "a": 3, "c": 4}), ["a", "b"]),
    (({}, {"d1": 1}), []),
]
run_checks(detect_sync_conflicts, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def detect_sync_conflicts_reference(source_a: dict[str, int], source_b: dict[str, int]) -> list[str]:
    shared = set(source_a) & set(source_b)
    return sorted(doc_id for doc_id in shared if source_a[doc_id] != source_b[doc_id])


assert run_checks(detect_sync_conflicts_reference, cases)

**Explain it in 30 seconds**

> "I only look at documents both sources know about, using a set intersection of their keys. For each one I compare the version numbers and keep the IDs that disagree, sorted so the output is stable. Documents on only one side are a different problem, missing or deleted, and I would report them separately. In production I would compare content hashes too, because timestamps and version counters can lie."

**Edge cases to mention**

- A document in only one source is not a conflict here. It needs its own missing or deleted report.
- Same version number with different content is not caught without a content hash.
- Deleted documents need a tombstone (a record that says "this was deleted"), or they look missing.
- A sync run that stopped halfway may show old versions for documents it never reached.

**If they ask "how would this work in production?"**

- Store a content hash next to the version, so real changes are caught even when versions agree.
- Use tombstones for deletes and checkpoints so a half-finished sync can resume where it stopped.
- Alert when the conflict rate jumps, because that usually means a broken connector.

## Problem 15: Duplicate Support Ticket Detection

**The situation**

A GenAI triage tool groups support tickets so one engineer can fix a shared issue once. Many tickets say the same thing with different casing and punctuation. Tickets from different customers must never be grouped, even when the words match.

**Your task**

Write `duplicate_ticket_pairs(tickets)`, where each ticket is `(tenant_id, title)`. Return `(first_index, later_index)` pairs for tickets in the same tenant whose titles match after cleaning.

**Example**

`[("t1", "Login not working!"), ("t1", "login not working")]` -> `[(0, 1)]`

**Ask before coding**

- What counts as a duplicate: the same words, or the same meaning?
- Should tickets from different customers ever match?
- Do you want pairs, or groups of duplicates?

**Plan to say aloud**

1. Clean each title: lowercase, keep only the words, join them with single spaces.
2. Skip titles that are empty after cleaning.
3. Build a key of (tenant_id, cleaned title).
4. Keep a dict from key to the first index seen.
5. When a key repeats, record (first index, current index).

### Prompt for your AI assistant

```text
Write a Python function `duplicate_ticket_pairs(tickets: list[tuple[str, str]]) -> list[tuple[int, int]]`. Each ticket is (tenant_id, title). Clean a title by lowercasing it and keeping only its words, joined by single spaces (use `re.findall` with the pattern `\w+`). Titles that are empty after cleaning are never duplicates. Two tickets are duplicates only if they have the same tenant_id and the same cleaned title. Return (index_of_first_occurrence, index_of_later_ticket) pairs in input order. Show only the function.
```

**Check the AI's code for**

- Leaves tenant_id out of the key, so two customers' tickets get merged.
- Pairs each copy with the previous copy, giving (2, 3) instead of (0, 3).
- Treats two empty or punctuation-only titles as duplicates of each other.
- Removes punctuation but not extra spaces, so `VPN   down` and `VPN down` do not match.

In [ ]:
import re


def duplicate_ticket_pairs(tickets: list[tuple[str, str]]) -> list[tuple[int, int]]:
    r"""Return (first_index, later_index) pairs of duplicate tickets.

    Rules:
    - Clean a title: lowercase, keep only words (re.findall with the pattern r"\w+"), join with one space.
    - Titles that are empty after cleaning are never duplicates.
    - Duplicates need the same tenant_id AND the same cleaned title.
    - Pair each later copy with the FIRST ticket that had that key.
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([("t1", "Login not working!"), ("t1", "login not working")],), [(0, 1)]),
    (([("t1", "Login failed"), ("t2", "login failed")],), []),
    (([("t1", "VPN down"), ("t1", "Printer jam"), ("t1", "vpn down"), ("t1", "VPN   DOWN!!")],), [(0, 2), (0, 3)]),
    (([("t1", "!!!"), ("t1", "")],), []),
    (([],), []),
]
run_checks(duplicate_ticket_pairs, cases)

### Reference solution

Try first. Then compare.

In [ ]:
import re


def duplicate_ticket_pairs_reference(tickets: list[tuple[str, str]]) -> list[tuple[int, int]]:
    first_seen: dict[tuple[str, str], int] = {}
    pairs = []
    for i, (tenant_id, title) in enumerate(tickets):
        cleaned = " ".join(re.findall(r"\w+", title.lower()))
        if not cleaned:
            continue  # nothing to compare
        key = (tenant_id, cleaned)
        if key in first_seen:
            pairs.append((first_seen[key], i))
        else:
            first_seen[key] = i
    return pairs


assert run_checks(duplicate_ticket_pairs_reference, cases)

**Explain it in 30 seconds**

> "I turn each title into a clean key: lowercase words with punctuation and extra spaces removed, plus the tenant ID. I remember the first ticket I saw for each key, and every later match is paired with that first one. Putting the tenant in the key means two customers are never merged. This is an exact-match baseline; for different wording with the same meaning I would add embeddings and a human check."

**Edge cases to mention**

- The same issue in different words, like "cannot sign in" and "login broken", is missed.
- The same words for different issues can be wrongly matched. Short titles make this worse.
- Empty or punctuation-only titles are skipped.
- Three or more copies all pair with the first one.

**If they ask "how would this work in production?"**

- Add embedding similarity with a confidence threshold for tickets worded differently.
- Compare only within a customer and a time window, so old tickets do not match new ones.
- Suggest merges for a human to approve, since merging is hard to undo, and track wrong merges as feedback.

## Wrap-up

- All three turn messy values into a fair comparison: cleaned strings, shared keys, or cleaned titles.
- Choosing which records to compare (union, intersection, same tenant) is the real decision. Say it out loud.
- They detect problems and return a stable, sorted report. Fixing or merging is a separate, approved step.